# FLARE22 预处理（Colab）

与 `colab_abdomenct1k_preprocess.ipynb`、`preprocess/proc_spatial_sequence.py` **相同的处理逻辑与超参**：
- 每个 nii volume：**50 个 npy**（z≥128 时）或 **33 个 npy**（z<128 时，`int(50/1.5)`）
- patch **128³**，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`

**输入**：`/content/drive/MyDrive/dataset/pretrain/FLARE22/unzip` 下递归收集所有 `.nii.gz`（可含子目录）；跳过 `__MACOSX`。

**输出**：所有 `.npy` 扁平放在 `/content/drive/MyDrive/dataset/pretrain/FLARE22/npy`；命名习惯与 AbdomenCT-1K / TCIA 一致：`{ds_name}_{nii_id}_{image_index}_{seq}.npy`。

In [ ]:
# Cell 1: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: 安装依赖
!pip install -q nibabel monai tqdm

In [ ]:
# Cell 3: FLARE22 预处理（与 proc_spatial_sequence / AbdomenCT-1K Colab 逻辑一致）
import os
import random
import numpy as np
import nibabel as nib
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

DRIVE_UNZIP = '/content/drive/MyDrive/dataset/pretrain/FLARE22/unzip'
DRIVE_SAVE_ROOT = '/content/drive/MyDrive/dataset/pretrain/FLARE22/npy'

PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode='trilinear'),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume_flare22(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size):
    image, affine = load_nii_data(image_file)
    path_parts = image_file.replace('\\', '/').split('/')
    ds_name = path_parts[-3] if len(path_parts) >= 3 else 'FLARE22'
    nii_id = path_parts[-1].split('.nii.gz')[0].split('.nii')[0].split('.mha')[0]

    if len(image.shape) == 4:
        return []
    images = [image]
    n, patch_path_list = 0, []

    for image_index, image in enumerate(images):
        image = image.transpose((2, 1, 0))
        _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
        for i in range(_patch_num):
            n += 1
            patch_size = random.choice(patch_size_list)
            image_patch, _ = cut_patch(image, patch_size)
            image_patch = image_patch.transpose((2, 1, 0))
            image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
            save_name = os.path.join(save_root, '%s_%s_%s_%d.nii.gz' % (ds_name, nii_id, image_index, start_index + n))
            patch_path_list.append(save_name)
            np.save(save_name.replace('.nii.gz', '.npy'), image_patch)
    return patch_path_list


def collect_flare22_nii_files(unzip_root):
    out = []
    if not os.path.isdir(unzip_root):
        return out
    for dirpath, _, files in os.walk(unzip_root):
        if '__MACOSX' in dirpath.replace('\\', '/'):
            continue
        for f in files:
            lower = f.lower()
            if lower.endswith('.nii.gz') or lower.endswith('.nii'):
                out.append(os.path.join(dirpath, f))
    return sorted(out)


os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
data_list = collect_flare22_nii_files(DRIVE_UNZIP)
print(f'unzip root: {DRIVE_UNZIP}')
print(f'found {len(data_list)} nii / nii.gz')

if len(data_list) == 0:
    print('No files found; check FLARE22 unzip path on Drive.')
else:
    patch_list_all = []
    for i, path in enumerate(tqdm(data_list, desc='preprocess')):
        patch_list = cut_and_save_one_volume_flare22(path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE)
        patch_list_all += patch_list
        if (i + 1) % 20 == 0:
            print(f'[{i+1}/{len(data_list)}] {path} -> {len(patch_list)} npy')
    print(f'\nDone. total {len(patch_list_all)} npy -> {DRIVE_SAVE_ROOT}')

In [ ]:
# Cell 4 (可选): 生成 train_patch_spatial.txt 用于 pretrain
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), 'pretrain_lists')
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith('.npy')])
with open(os.path.join(list_save_dir, 'train_patch_spatial.txt'), 'w') as f:
    f.write('\n'.join(npy_files))
print(f'train_patch_spatial.txt -> {list_save_dir}')